# Rozwiązywanie układów równań liniowych w NumPy i SymPy

Układ równań liniowych można zapisać w postaci macierzowej $A\mathbf{x}=\mathbf{b}$. Pokażemy, jak rozwiązać taki układ numerycznie za pomocą NumPy oraz dokładnie, symbolicznie za pomocą SymPy. Na końcu sprawdzimy, jak SymPy opisuje układ mający nieskończenie wiele rozwiązań.

## Układ z jednym rozwiązaniem

Rozważmy układ

$$\begin{aligned}
2x+y-z &= 8,\\
-3x-y+2z &= -11,\\
-2x+y+2z &= -3.
\end{aligned}$$

Współczynniki przy niewiadomych tworzą macierz $A$, a prawe strony równań — wektor $\mathbf{b}$.

In [1]:
import numpy as np
import sympy as sp

A_np = np.array([
    [2, 1, -1],
    [-3, -1, 2],
    [-2, 1, 2],
])
b_np = np.array([8, -11, -3])

print("Macierz współczynników A:")
print(A_np)
print("Wektor prawych stron b:")
print(b_np)

Macierz współczynników A:
[[ 2  1 -1]
 [-3 -1  2]
 [-2  1  2]]
Wektor prawych stron b:
[  8 -11  -3]


## Rozwiązanie numeryczne w NumPy

Dla kwadratowego układu z jednym rozwiązaniem używamy `numpy.linalg.solve(A, b)`. Nie obliczamy jawnie macierzy odwrotnej: `solve` rozwiązuje układ bezpośrednio i jest do tego właściwą funkcją. Wynik jest numeryczny (zwykle typu zmiennoprzecinkowego), nawet gdy dane wejściowe są całkowite.

In [2]:
x_np = np.linalg.solve(A_np, b_np)

print("Rozwiązanie [x, y, z]:", x_np)
print("Sprawdzenie A @ x = b:", np.allclose(A_np @ x_np, b_np))
print("Reszta A @ x - b:", A_np @ x_np - b_np)

Rozwiązanie [x, y, z]: [ 2.  3. -1.]
Sprawdzenie A @ x = b: True
Reszta A @ x - b: [-8.88178420e-16  1.77635684e-15  0.00000000e+00]


## Rozwiązanie dokładne w SymPy

SymPy przechowuje wartości dokładne i może zwrócić rozwiązanie w postaci symbolicznej. Dla macierzowego zapisu $A\mathbf{x}=\mathbf{b}$ metodą `LUsolve` rozwiązujemy układ bez wyznaczania macierzy odwrotnej. Zapisujemy dane jako macierze SymPy, aby wynik pozostał dokładny.

In [3]:
A = sp.Matrix([
    [2, 1, -1],
    [-3, -1, 2],
    [-2, 1, 2],
])
b = sp.Matrix([8, -11, -3])

x_sym = A.LUsolve(b)
print("Rozwiązanie [x, y, z]:")
sp.pprint(x_sym)
print("Sprawdzenie A * x - b:")
sp.pprint(A * x_sym - b)

Rozwiązanie [x, y, z]:
⎡2 ⎤
⎢  ⎥
⎢3 ⎥
⎢  ⎥
⎣-1⎦
Sprawdzenie A * x - b:
⎡0⎤
⎢ ⎥
⎢0⎥
⎢ ⎥
⎣0⎦


Jeśli chcemy opisać niewiadome jako symbole, możemy użyć `linsolve`. Funkcja ta zwraca zbiór rozwiązań — także wtedy, gdy jest ich nieskończenie wiele albo nie ma żadnego.

In [4]:
x, y, z = sp.symbols("x y z")
rozwiazanie = sp.linsolve((A, b), (x, y, z))
print("Zbiór rozwiązań:")
sp.pprint(rozwiazanie)

Zbiór rozwiązań:
{(2, 3, -1)}


## Układ z nieskończenie wieloma rozwiązaniami

Macierz osobliwa nie musi oznaczać braku rozwiązań — układ może być nieoznaczony. `linsolve` zwróci wtedy rozwiązanie parametryczne. Poniższe dwa równania opisują tę samą prostą $x+y=2$.

In [5]:
A_nieoznaczony = sp.Matrix([
    [1, 1],
    [2, 2],
])
b_nieoznaczony = sp.Matrix([2, 4])

sp.pprint(sp.linsolve((A_nieoznaczony, b_nieoznaczony), (x, y)))

{(2 - y, y)}


## Kiedy użyć której biblioteki?

- **NumPy** — do szybkich obliczeń numerycznych na danych liczbowych. `np.linalg.solve` wymaga macierzy kwadratowej i układu z jednym rozwiązaniem; dla układów osobliwych lub źle uwarunkowanych może zgłosić `LinAlgError`. Dla problemu najmniejszych kwadratów (np. układu nadokreślonego) służy `np.linalg.lstsq`.
- **SymPy** — do rachunków dokładnych, symbolicznych oraz badania liczby rozwiązań. `LUsolve` jest wygodne dla układu o jednym rozwiązaniu, a `linsolve` zwraca zbiór rozwiązań również dla układów nieoznaczonych i sprzecznych.

W obliczeniach numerycznych sprawdzamy wynik z tolerancją (np. przez `np.allclose`), ponieważ zaokrąglenia zmiennoprzecinkowe mogą powodować niewielkie różnice.